## Overview: data preparation for project-level regressions

* Regression preparations 
    * Outcomes: number of commits a project gets, number of different contributors (also ru/ua), number of quality-weighted contributors (also ru/ua), number of stars (!ru, ua), number of forks (!ru, ua)
    * Level: monthly (aggregated to quarterly in stata)
    * Controls: project age, language, pre-2014 stars and forks, quality of owner (pre-2014) (owner's number of stars and commits), number of contributors, number of locations of contributors  

In [ ]:
import os, sys
import pandas as pd
import sqlite3
import csv

In [ ]:
# Set path
ROOT = Path(__file__).resolve().parents[2]

DATA = ROOT / "database"
EXTRACTIONS = DATA / "extractions"
STATA = DATA / "stata"
OUTPUT = ROOT / "output"
FIGURES = OUTPUT / "figures"
TABLES = OUTPUT / "tables"

# Credentials 
credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")

if not credentials_path:
    raise EnvironmentError(
        "GOOGLE_APPLICATION_CREDENTIALS is not set. "
        "Please set it to the path of your Google Cloud credentials JSON file."
    )

credentials_path = Path(credentials_path).expanduser().resolve()

if not credentials_path.exists():
    raise FileNotFoundError(
        f"Google Cloud credentials file not found: {credentials_path}"
    )

In [ ]:
# settings for bigquery 
dry = bigquery.QueryJobConfig(dry_run = True, use_query_cache=False) 
run = bigquery.QueryJobConfig(dry_run=False, use_query_cache=True)


In [ ]:
# project level commits 
# number of commits by project-id and month: commits_all, commits_self, commits_ua, commits_ru
# only projects that existed prior to 2014
query = '''
DROP VIEW IF EXISTS `friends-foes.projects.project_commits`;
CREATE VIEW `friends-foes.projects.project_commits`
AS 
# all commits that a project recevied 
SELECT all_c.year as year, all_c.month as month, all_c.project_id as project_id, owner_id, forked, country_owner, 
        dlk_owner, language, project_year, project_month, company13, company_current, n_authors, n_author_countries,
        commits_all_int, commits_all_ext, commits_nself_int, commits_nself_ext, commits_ru_int, commits_ru_ext, commits_ua_int, commits_ua_ext 
 FROM 
(SELECT year, month, project_id, owner_id, forked, country_owner, dlk_owner, 
       SUM(commits) AS commits_all_int, COUNT(commits) AS commits_all_ext, 
       COUNT(DISTINCT author_id) AS n_authors, 
       COUNT(DISTINCT country_author) AS n_author_countries, 
       SUM(IF(country_author <> country_owner, commits, NULL)) as commits_nself_int, 
       COUNT(IF(country_author <> country_owner, commits, NULL)) as commits_nself_ext, 
       SUM(IF(country_author = 'ru', commits, NULL)) as commits_ru_int, 
       COUNT(IF(country_author = 'ru', commits, NULL)) as commits_ru_ext, 
       SUM(IF(country_author = 'ua', commits, NULL)) as commits_ua_int, 
       COUNT(IF(country_author = 'ua', commits, NULL)) as commits_ua_ext 
FROM `friends-foes.commits.puc`
WHERE (project_id IN (SELECT DISTINCT project_id FROM `friends-foes.commits.puc` WHERE year < 2014))
GROUP BY year, month, project_id, owner_id, country_owner, dlk_owner,forked
ORDER BY project_id, year, month, owner_id) all_c

# merge to static information (merge on project_id)
LEFT JOIN 
(SELECT DISTINCT EXTRACT(YEAR FROM created_at) as project_year, EXTRACT(MONTH FROM created_at) as project_month,
 language, id  FROM `friends-foes.commits.projects`) info
ON info.id = all_c.project_id

# merge to information if an owner is affiliated to a company
LEFT JOIN (SELECT DISTINCT id, year, company13, company_current FROM `friends-foes.user_panel.sample_companies`) comp
ON comp.id = all_c.owner_id

'''
client.query(query, job_config=run) 

In [ ]:
# project level quality: number of stars and forks 
query = '''
DROP VIEW IF EXISTS `friends-foes.projects.quality`; 

# cumulative is only needed for cross-check 
CREATE VIEW  `friends-foes.projects.quality` 
AS 
#
SELECT DISTINCT project_id, year, month, project_stars, 
       SUM(project_stars) OVER(PARTITION BY project_id ORDER BY DATE(year, month, 01)) AS project_stars_cum
FROM 
  (
  SELECT repo_id as project_id,
        EXTRACT(YEAR FROM created_at) AS year,
        EXTRACT(MONTH FROM created_at) AS month, 
        COUNT(*) AS project_stars  
    FROM `ghtorrentmysql1906.MySQL1906.watchers` s
    LEFT JOIN 
      (SELECT id, country_current FROM `friends-foes.user_panel.sample_locations_panel` WHERE year = 2019) u
    ON u.id = s.user_id 
    WHERE country_current NOT IN ('ru', 'ua') OR country_current IS NULL
    GROUP BY project_id,year, month
  )
'''
client.query(query, job_config=run)   

query = '''
DROP VIEW IF EXISTS `friends-foes.projects.forks`; 

CREATE VIEW  `friends-foes.projects.forks` 
AS 
#
SELECT DISTINCT forked_from as project_id, year, month, project_forks
FROM 
  (SELECT forked_from,
        EXTRACT(YEAR FROM created_at) AS year,
        EXTRACT(MONTH FROM created_at) AS month, 
        COUNT(*) AS project_forks  
    FROM `ghtorrentmysql1906.MySQL1906.projects` p
    LEFT JOIN 
      (SELECT id, country_current FROM `friends-foes.user_panel.sample_locations_panel` WHERE year = 2019) u
    ON p.owner_id = u.id 
    WHERE 
    (forked_from IN (SELECT DISTINCT project_id FROM `friends-foes.projects.project_commits`) ) AND
    (country_current NOT IN ('ru', 'ua') OR country_current IS NULL)
    GROUP BY forked_from, year, month
  )
'''
client.query(query, job_config=run)   

In [ ]:
# project's pre-2014 characteristics 
query = '''
DROP VIEW IF EXISTS `friends-foes.projects.pre`; 

CREATE VIEW  `friends-foes.projects.pre` 
AS 
SELECT c.project_id, commits_int_pre, commits_ext_pre, n_authors_pre, n_author_countries_pre,
       project_stars_pre, project_forks_pre, 
       owner_commits_pre, owner_projects_pre, owner_contributors_pre, owner_stars_pre
FROM 
(
(SELECT DISTINCT project_id, SUM(commits_nself_int) as commits_int_pre, SUM(commits_nself_ext) as commits_ext_pre, 
        AVG(n_authors) AS n_authors_pre, 
        AVG(n_author_countries) AS n_author_countries_pre
        FROM `friends-foes.projects.project_commits`
        WHERE year < 2014
        GROUP BY project_id) c
LEFT JOIN 
(SELECT DISTINCT project_id, SUM(project_stars) as project_stars_pre
        FROM `friends-foes.projects.quality`
        WHERE year < 2014
        GROUP BY project_id) s
ON c.project_id = s.project_id
LEFT JOIN 
(SELECT DISTINCT project_id, SUM(project_forks) as project_forks_pre
        FROM `friends-foes.projects.forks`
        WHERE year < 2014
        GROUP BY project_id) f
ON c.project_id = f.project_id
LEFT JOIN 
(SELECT DISTINCT project_id, commits_pre as owner_commits_pre, projects_pre as owner_projects_pre, 
        contributors_pre as owner_contributors_pre, user_stars_cum as owner_stars_pre
        FROM `friends-foes.projects.project_commits` a
        LEFT JOIN `friends-foes.quality.user_quality` b ON a.owner_id = b.user_id) o
ON c.project_id = o.project_id
)

'''
client.query(query, job_config=run)

In [ ]:
# join three views and generate a table
query = '''
DROP TABLE IF EXISTS `friends-foes.projects.merged`; 

CREATE TABLE `friends-foes.projects.merged`
AS SELECT a.project_id, a.year, a.month, owner_id, forked, country_owner, 
        dlk_owner, language, project_year, project_month, company13, company_current, 
        n_authors, n_author_countries,
        commits_all_int, commits_all_ext, commits_nself_int, commits_nself_ext, 
        commits_ru_int, commits_ru_ext, commits_ua_int, commits_ua_ext, 
        project_stars, project_forks, 
        commits_int_pre, commits_ext_pre, n_authors_pre, n_author_countries_pre,
        project_stars_pre, project_forks_pre, 
        owner_commits_pre, owner_projects_pre, owner_contributors_pre, owner_stars_pre
FROM `friends-foes.projects.project_commits` a 
LEFT JOIN `friends-foes.projects.quality` b ON a.project_id = b.project_id and a.year = b.year and a.month = b.month
LEFT JOIN `friends-foes.projects.forks` c ON a.project_id = c.project_id and a.year = c.year and a.month = c.month
LEFT JOIN `friends-foes.projects.pre` d ON a.project_id = d.project_id

'''
client.query(query, job_config=run)

In [ ]:
# treated view 
# projects that had an ua-ru collaboration in 2013
# only projects with at least 2 different authors and 2 different countries 
#( only non-forked projects)
# only those where project_year < 2014
query = '''
DROP VIEW IF EXISTS `friends-foes.projects.treated`; 
CREATE VIEW `friends-foes.projects.treated`
AS 
       
SELECT  DISTINCT a.project_id as project_id, owner_id, country_owner, 
        dlk_owner, forked, language, project_year, project_month, company13, company_current, n_authors, n_author_countries,
        commits_nself_int, commits_nself_ext, commits_ru_int, commits_ru_ext, commits_ua_int, commits_ua_ext, 
        CASE WHEN commits_ru_ext*commits_ua_ext > 0 THEN 1 ELSE 0
        END treated, 
        commits_int_pre, commits_ext_pre, project_stars_pre, project_forks_pre, 
        owner_commits_pre, owner_projects_pre, owner_contributors_pre, owner_stars_pre 
 FROM 
 # only projects that were active in 2013, had at least two authors and at least two different countries ( no forks)
(SELECT project_id, owner_id, country_owner, dlk_owner, forked,
       COUNT(DISTINCT author_id) AS n_authors, 
       COUNT(DISTINCT country_author) AS n_author_countries, 
       SUM(IF(country_author <> country_owner, commits, NULL)) as commits_nself_int, 
       COUNT(IF(country_author <> country_owner, commits, NULL)) as commits_nself_ext, 
       SUM(IF(country_author = 'ru', commits, NULL)) as commits_ru_int, 
       COUNT(IF(country_author = 'ru', commits, NULL)) as commits_ru_ext, 
       SUM(IF(country_author = 'ua', commits, NULL)) as commits_ua_int, 
       COUNT(IF(country_author = 'ua', commits, NULL)) as commits_ua_ext 
FROM `friends-foes.commits.puc`
WHERE year = 2013 
GROUP BY project_id, owner_id, country_owner, dlk_owner, forked
HAVING commits_nself_int > 1 AND n_author_countries>1) a

# merge to static information (merge on project_id)
LEFT JOIN 
(SELECT DISTINCT EXTRACT(YEAR FROM created_at) as project_year, EXTRACT(MONTH FROM created_at) as project_month,
 language, id  FROM `friends-foes.commits.projects`) info
ON info.id = a.project_id

# merge to information if an owner is affiliated to a company
LEFT JOIN (SELECT DISTINCT id, company13, company_current FROM `friends-foes.user_panel.sample_companies` WHERE year = 2013) comp
ON comp.id = a.owner_id

# project's qulaity
LEFT JOIN `friends-foes.projects.pre` d ON a.project_id = d.project_id

'''
client.query(query, job_config=run)

In [ ]:
# extract the table to select treated projects and a matched group of similar ones
query='''
SELECT * 
FROM `friends-foes.projects.treated`
#WHERE project_year < 2014
'''

job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'treated_projects.csv', encoding = 'utf-8', index=False)

In [ ]:
query = '''
SELECT project_id, year, month, company_current, 
        n_authors, n_author_countries,
        commits_all_int, commits_all_ext, commits_nself_int, commits_nself_ext, 
        commits_ru_int, commits_ru_ext, commits_ua_int, commits_ua_ext, 
        project_stars, project_forks
FROM `friends-foes.projects.merged`
WHERE project_id IN (SELECT DISTINCT project_id FROM `friends-foes.projects.treated`)
'''
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
df = job.to_dataframe()  
df.to_csv(STATA / 'perf_projects.csv', encoding = 'utf-8', index=False)